# Build cloud-hosted tables

Create your own hosted database, declare a multimodal schema in it, insert postcards, and watch
the database compute derived columns and split a voice note into segments. No model download and
no provider key. The one slow part is the first image build, which takes 10+ minutes.

Set your database. It must match the `[[tool.pixeltable.database]]` entry in `pyproject.toml`.

In [1]:
# Your hosted database. Set your org slug (see `pxt org list`).
# This MUST match the [[tool.pixeltable.database]] entry in pyproject.toml.
DB = 'pxt://pixeltable:champs'  # <-- edit this
print('targeting', DB)

targeting pxt://pixeltable:champs


## Create and ship your database

`pyproject.toml` declares the database; `pxt db update` creates it and builds a hosted image from
your `uv.lock`, so the database runs your project. `diff` previews, `update` applies. The first
build takes 10+ minutes; later edits ship in seconds. Wait for `AVAILABLE`.

In [2]:
!pxt db diff   $DB
!pxt db update $DB          # first run: create the db and build the image (10+ min)
!pxt db status $DB --json   # wait for state AVAILABLE

pxt: 422 no [[pixeltable.database]] entry names 'pxt://pixeltable:champs'; add one to /Users/alison-pxt/Documents/Github/pxt-champs/pyproject.toml:
  [[pixeltable.database]]
  name = 'pxt://pixeltable:champs'
pxt: 422 no [[pixeltable.database]] entry names 'pxt://pixeltable:champs'; add one to /Users/alison-pxt/Documents/Github/pxt-champs/pyproject.toml:
  [[pixeltable.database]]
  name = 'pxt://pixeltable:champs'
{"org_id": "org_01K34P4S4ETYS15YDP4WTSMPTE", "db_slug": "champs", "default_bucket": "home", "db_name": "champs", "state": "AVAILABLE", "location": "", "cluster": "", "runtime_image": "339712970370.dkr.ecr.us-east-1.amazonaws.com/pxt_cloud/db-runtime/prod:pixeltable-champs-39929129", "last_build_state": "ACTIVE", "update_runtime_status": {"request_id": "f44f76dd-57fa-4a0a-a321-b3be44900bc6", "build_id": "39929129-d2fe-4202-80a0-72e9d234cccb", "requested_by": "user_01K3H1PB7FYTJH17EF5WFXVV05", "stage": "DEPLOY", "state": "SUCCEEDED", "error": null, "requested_at": 1788380922.96

## Write and apply your schema

A **schema** is the shape and logic of your data: which tables exist, each table's columns and
their types, and how the derived columns are computed from other columns. You write it as a
class-based Python file. Yours is `app.py`, a `Postcards` table plus a `Segments` view. Read it:

In [ ]:
!cat app.py

`check` validates the file, `diff` shows what would change, and `update` applies it. Then
`describe` prints the result, with each column's type and the expression it is computed from.

In [ ]:
!pxt schema check  app.py      # is the file valid?
!pxt schema diff   app.py $DB   # what would change?
!pxt schema update app.py $DB   # create the tables and the view

!pxt describe $DB/postcards
!pxt describe $DB/segments      # the view, plus the base columns it carries

## Insert postcards, and the database computes

Each insert is one transaction. When it returns, the computed columns are filled and the
`Segments` view already holds the audio the database split out, with no pipeline code of yours.

In [ ]:
import pixeltable as pxt

postcards = pxt.get_table(f'{DB}/postcards')
postcards.insert([
    {'caption': 'wish you were here',       'image': 'data/beach.jpg',  'voice': 'data/beach.wav'},
    {'caption': 'greetings from the pines', 'image': 'data/forest.jpg', 'voice': 'data/forest.wav'},
    {'caption': 'hot out here',             'image': 'data/desert.jpg', 'voice': 'data/desert.wav'},
])

postcards.select(postcards.caption_upper, postcards.width, postcards.thumb).collect()

The view exploded each voice note into segments:

In [ ]:
segments = pxt.get_table(f'{DB}/segments')
segments.select(segments.segment_start, segments.segment_end, segments.seconds).collect()

## See it

In [ ]:
!pxt ls --tree $DB   # postcards (table), segments (view)
!pxt dashboard       # browse it, with images and audio inline

## Build your own

Scaffold a schema with the CLI: `pxt schema example` prints a full one, `--brief` a minimal one,
and `--out my_app.py` writes it to edit and apply. `pxt service example` does the same for routes.

In [ ]:
!pxt schema example --brief

## What next

- `03_serve_api.ipynb` serves this database as an API.
- `04_image_search.ipynb` searches images by text with CLIP (optional, installs a large model).
- `05_share_table.ipynb` shares a table (optional).

Stuck? `TROUBLESHOOTING.md`.